# 01 · Llamando a un LLM, paso a paso

**Objetivo.** Ver las piezas de una llamada a un modelo: mensajes, parámetros, petición HTTP y lectura de la respuesta. Primero con un modelo local (LM Studio) y después con Gemini, para comprobar que la idea es la misma aunque cambie el proveedor.

## 0. Preparación

Las llaves viven en `.env` y la configuración en `config/`; nunca se escriben en el notebook. LM Studio debe estar abierto, con un modelo cargado y el servidor local activo.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from receipt_validation.config import load_settings

settings = load_settings(ROOT)
print(f"Project root: {ROOT}")

Project root: /Users/andrei/Documents/Projects/Curso LLM Codes/session_2_architecture


In [2]:
import requests

LOCAL_URL = f"{settings['lmstudio_base_url'].rstrip('/')}/chat/completions"
LOCAL_MODEL = settings["default_lmstudio_model"]
GEMINI_MODEL = settings["default_gemini_model"]
GEMINI_URL = settings['gemini_base_url']

PROMPT = "Invent a name and a one-sentence slogan for a coffee shop on the Moon."

print("LOCAL MODEL - URL:", f"{LOCAL_MODEL} - {LOCAL_URL}")
print("GEMINI MODEL - URL:", f"{GEMINI_MODEL} - {GEMINI_URL}")

LOCAL MODEL - URL: google/gemma-4-e2b - http://localhost:1234/v1/chat/completions
GEMINI MODEL - URL: gemini-3.1-flash-lite - https://generativelanguage.googleapis.com/v1beta/interactions


## 1. Anatomía de una llamada

Una llamada se arma por bloques. El primero es la **conversación**: una lista de mensajes con roles. `system` define el comportamiento del modelo y `user` lleva la pregunta.

In [3]:
messages = [
    {"role": "system", "content": "You are a creative copywriter. Answer in one short line."},
    {"role": "user", "content": PROMPT},
]
messages

[{'role': 'system',
  'content': 'You are a creative copywriter. Answer in one short line.'},
 {'role': 'user',
  'content': 'Invent a name and a one-sentence slogan for a coffee shop on the Moon.'}]

El segundo bloque son los **parámetros**: qué modelo usar, cuánta variación permitir (`temperature`) y el máximo de tokens de salida (`max_tokens`).

In [4]:
payload = {
    "model": LOCAL_MODEL,
    "messages": messages,
    "temperature": 0.7,
    "max_tokens": 600,
}
payload

{'model': 'google/gemma-4-e2b',
 'messages': [{'role': 'system',
   'content': 'You are a creative copywriter. Answer in one short line.'},
  {'role': 'user',
   'content': 'Invent a name and a one-sentence slogan for a coffee shop on the Moon.'}],
 'temperature': 0.7,
 'max_tokens': 600}

El tercer bloque es la **petición HTTP**. El servidor responde un JSON con la respuesta, el razonamiento (si el modelo razona) y el conteo de tokens.

In [5]:
response = requests.post(LOCAL_URL, json=payload, timeout=120)
data = response.json()
data

{'id': 'chatcmpl-wy4wid82uc65ktanuuipe',
 'object': 'chat.completion',
 'created': 1783731189,
 'model': 'google/gemma-4-e2b',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Lunar Roast: Where every cup orbits wonder.',
    'reasoning_content': "\nThinking Process:\n\n1.  **Analyze the Request:** Invent a name and a one-sentence slogan for a coffee shop located on the Moon.\n2.  **Identify Key Themes:** Coffee/Brewing, Space/Moon, Isolation/Adventure, Cosmic/Otherworldly.\n3.  **Brainstorm Names (Focusing on Moon/Space):** Lunar, Crater, Selene, Astro, Moonbeam, Gravity-defying.\n4.  **Brainstorm Slogans (Focusing on Experience/Coffee):** Must be short, evocative, and relate to the unique location.\n5.  **Combine and Refine (Goal: Short, Creative Copy):**\n\n    *   *Attempt 1:* Name: Lunar Brew. Slogan: Taste the silence of space. (A bit too generic.)\n    *   *Attempt 2:* Name: Selene's Sip. Slogan: Your daily dose of celestial calm. (Better.)\n    *   

Último bloque: **leer la respuesta**. El texto vive en `choices[0].message.content`, el razonamiento en `reasoning_content` y los tokens en `usage`.

In [6]:
message = data["choices"][0]["message"]
answer = message.get("content") or ""
reasoning = message.get("reasoning_content") or ""
usage = data.get("usage", {})

Con un helper pequeño mostramos los resultados de forma legible en vez de JSON crudo.

In [7]:
from IPython.display import Markdown, display

def show(title, body):
    display(Markdown(f"#### {title}\n\n{body}\n\n---"))

show("Answer", answer)
if reasoning:
    show("Reasoning", reasoning)
show("Tokens", f"input: {usage.get('prompt_tokens')} · output: {usage.get('completion_tokens')}")

#### Answer

Lunar Roast: Where every cup orbits wonder.

---

#### Reasoning


Thinking Process:

1.  **Analyze the Request:** Invent a name and a one-sentence slogan for a coffee shop located on the Moon.
2.  **Identify Key Themes:** Coffee/Brewing, Space/Moon, Isolation/Adventure, Cosmic/Otherworldly.
3.  **Brainstorm Names (Focusing on Moon/Space):** Lunar, Crater, Selene, Astro, Moonbeam, Gravity-defying.
4.  **Brainstorm Slogans (Focusing on Experience/Coffee):** Must be short, evocative, and relate to the unique location.
5.  **Combine and Refine (Goal: Short, Creative Copy):**

    *   *Attempt 1:* Name: Lunar Brew. Slogan: Taste the silence of space. (A bit too generic.)
    *   *Attempt 2:* Name: Selene's Sip. Slogan: Your daily dose of celestial calm. (Better.)
    *   *Attempt 3:* Name: Crater Coffee. Slogan: Brewed under a billion stars. (Stronger imagery.)

6.  **Select the Best Option and Format (One short line):** I need to choose a name and a slogan that sound compelling.

7.  **Final Polish:** (Choosing a punchy, evocative option.)

---

#### Tokens

input: 46 · output: 290

---

## 2. Una función reutilizable

Los cuatro bloques anteriores caben en una sola función. A partir de aquí, cada experimento es una línea.

In [8]:
def call_local(prompt, temperature=0.7, max_tokens=600):
    payload = {
        "model": LOCAL_MODEL,
        "messages": [
            {"role": "system", "content": "You are a creative copywriter. Answer in one short line."},
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    response = requests.post(LOCAL_URL, json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    message = data["choices"][0]["message"]
    return {
        "answer": message.get("content") or "",
        "reasoning": message.get("reasoning_content") or "",
        "usage": data.get("usage", {}),
    }

## 3. Experimento: temperatura

Tarea creativa (nombre + eslogan) y **dos corridas por temperatura**. Con `0.0` las dos corridas salen casi idénticas: el modelo siempre elige el token más probable. Al subir la temperatura, cada corrida produce ideas distintas. Más diversidad, no «mejor».

> **Pausa:** antes de ejecutar, predigan qué pasa al repetir la llamada con `0.0` y con `1.2`.

In [9]:
for temperature in (0.0, 0.7, 1.2):
    for run in (1, 2):
        result = call_local(PROMPT, temperature=temperature)
        show(f"Temperature {temperature} · run {run}", result["answer"] or result["reasoning"])

#### Temperature 0.0 · run 1

Lunar Brews: Where every cup tastes like a cosmic journey.

---

#### Temperature 0.0 · run 2

Lunar Brews: Where every cup tastes like a cosmic journey.

---

#### Temperature 0.7 · run 1

Crater Coffee: Brewed under the silent gaze of a million stars.

---

#### Temperature 0.7 · run 2

Selene's Sip: Where cosmic clarity meets your morning ritual.

---

#### Temperature 1.2 · run 1

Luna Roast: Where every cup orbits a new world.

---

#### Temperature 1.2 · run 2

Lunar Latte: Brewed on the edge of forever.

---

## 4. Cómo razona el modelo

Los modelos de razonamiento «piensan» antes de responder y ese proceso llega en `reasoning_content`, separado de la respuesta final. Verlo ayuda a entender qué hace el modelo con la instrucción.

In [10]:
result = call_local("How many r's are in the word strawberry?")
show("Reasoning", result["reasoning"])
show("Answer", result["answer"])

#### Reasoning


Thinking Process:
1.  **Analyze the Request:** The user is asking for the number of 'r's in the word "strawberry".
2.  **Analyze the Role:** I am a creative copywriter and must answer in one short line.
3.  **Execute the Calculation:** S-T-R-A-W-B-E-R-R-Y.
    *   S-T-R (1) - A - W - B - E - R (2) - R (3) - Y.
    *   The word has three 'r's.
4.  **Formulate the Copywriter Response:** The answer must be direct and concise.

5.  **Final Output Generation:** (Focusing on the count)

---

#### Answer

Three r's make it a sweet trio!

---

## 5. Lo mismo con Gemini

Cambia la forma: el API de *interactions* recibe `input`, `system_instruction` y `generation_config`, la llave viaja en el header `x-goog-api-key`, y la respuesta llega en `steps` (el paso `model_output` trae el texto). Las piezas siguen siendo las mismas: mensajes, parámetros, petición y lectura. Por eso podemos repetir el experimento y comparar proveedores.

In [11]:
def call_gemini(prompt, temperature=0.7, max_tokens=600):
    payload = {
        "model": GEMINI_MODEL,
        "input": prompt,
        "system_instruction": "You are a creative copywriter. Answer in one short line.",
        "generation_config": {"temperature": temperature, "max_output_tokens": max_tokens},
    }
    headers = {"x-goog-api-key": settings["google_api_key"]}
    response = requests.post(GEMINI_URL, headers=headers, json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    answer = "".join(
        item["text"]
        for step in data["steps"] if step["type"] == "model_output"
        for item in step["content"] if item["type"] == "text"
    )
    return {"answer": answer, "usage": data.get("usage", {})}

In [12]:
for temperature in (0.0, 0.7, 1.2):
    for run in (1, 2):
        result = call_gemini(PROMPT, temperature=temperature)
        show(f"Gemini · temperature {temperature} · run {run}", result["answer"])

#### Gemini · temperature 0.0 · run 1

**Lunar Brew:** One small sip for man, one giant leap for your morning.

---

#### Gemini · temperature 0.0 · run 2

**Lunar Brew:** One small sip for man, one giant leap for your morning.

---

#### Gemini · temperature 0.7 · run 1

**Lunar Brew:** One small sip for man, one giant leap for your morning.

---

#### Gemini · temperature 0.7 · run 2

**Lunar Brew:** One small sip for man, one giant leap for your morning.

---

#### Gemini · temperature 1.2 · run 1

**Lunar Brew**: A giant leap in every cup.

---

#### Gemini · temperature 1.2 · run 2

**Lunar Brews:** One small sip for man, one giant leap for your morning.

---

## 6. Lo mismo con la librería oficial (`google-genai`)

En producción rara vez armamos el REST a mano: los proveedores publican SDKs que esconden la plomería (URL, headers, parseo). La llamada es la misma de siempre — modelo, instrucción de sistema, parámetros y contenido — pero en tres líneas. La llave la toma del entorno que ya cargamos desde `.env`.

In [13]:
from google import genai

client = genai.Client(api_key=settings["google_api_key"])

response = client.models.generate_content(
    model=GEMINI_MODEL,
    config={
        "system_instruction": "You are a creative copywriter. Answer in one short line.",
        "temperature": 0.7,
        "max_output_tokens": 600,
    },
    contents=PROMPT,
)

show("Gemini SDK", response.text)

#### Gemini SDK

**Lunar Brew:** One small sip for man, one giant leap for your morning.

---

## 7. Multimodal: transcribir un audio

Los modelos actuales no solo reciben texto. Con el mismo SDK subimos un archivo de audio a la File API y lo pasamos como contenido junto con la instrucción. Primero lo escuchamos, luego se lo damos al modelo.

In [14]:
from IPython.display import Audio

audio_path = ROOT / "data" / "audios" / "audio_shut_down_computer.mp3"
Audio(str(audio_path))

In [15]:
uploaded_file = client.files.upload(file=audio_path)
# https://pixabay.com/es/sound-effects/gente-shut-down-your-computer-ai-mee-universe-142703/

prompt = "Generate a transcript of the audio."

response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[prompt, uploaded_file],
)

show("Transcription", response.text)

#### Transcription

Do you want to shut down your computer?

---

## 8. Cierre

- Toda llamada tiene las mismas piezas: mensajes, parámetros, petición y lectura.
- `temperature` controla diversidad, no calidad.
- `max_tokens` es un límite de salida, no una meta.
- Cambiar de proveedor solo cambia la forma del payload, no el experimento.

<details><summary><strong>Chequeo para revelar al final</strong></summary>

1. ¿En qué campo llega la respuesta en cada API?
2. ¿Dónde aparecen los tokens de entrada y salida?
3. ¿Qué error esperan si la llave o el servidor local no están disponibles?
</details>